# Maia Corpus 0.3.0 — Builder v5: acquisition, curation & diversity

**Objective:** inventory authentic PT/EN/ES sources without confusing *available* with *approved* data; acquire supplementary language/educational texts; assess dialogue gaps; clean, segment, deduplicate, split, and export **only reviewed sources**.

**Scientific constraints:** No SFT, no model training, no automatic rights approval. Token counts are **GPT-2 proxies** until the final multilingual tokenizer is frozen. Preserve prior `corpus/` and `corpus_v4/` outputs.

**Source notes:** Wikipedia/Wikibooks Wikimedia licenses and page-specific exceptions require review; Gutenberg works and translations require jurisdiction-specific checks. The Wikimedia and Hugging Face terms are not blanket permission for every item. Dialogue extracted from dramatic works is *scripted dialogue*, not authentic contemporary chat.

References: [Wikisource reuse](https://en.wikisource.org/wiki/Wikisource:Reusing_Wikisource_content), [Wikimedia datasets](https://huggingface.co/docs/datasets/stream), [OPUS Books restrictions](https://huggingface.co/datasets/Helsinki-NLP/opus_books) (not automatically included).

In [ ]:
# 1. Setup
%pip -q install transformers tokenizers requests pandas matplotlib


In [ ]:
# 2. Configuration and Drive
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import os, re, json, csv, time, hashlib, unicodedata, random, shutil
import requests
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
except ImportError:
    BASE = Path.cwd() / 'Maia_Lite_0_3_0_Chat'

V5 = BASE / 'corpus_v5'
RAW = V5 / 'inbox'
REPORT = V5 / 'reports'
OUT = V5 / 'curated'
for d in (RAW, REPORT, OUT): d.mkdir(parents=True, exist_ok=True)
PREVIOUS = [BASE / 'corpus_v4/inbox', BASE / 'corpus/inbox']
LANGS = ('pt', 'en', 'es')
DOMAINS = ('language', 'dialogue', 'logic', 'educational', 'creative')
TARGET_DOMAIN = {'language':.30, 'dialogue':.20, 'logic':.25, 'educational':.15, 'creative':.10}
TARGET_LANG = {'pt':.40, 'en':.35, 'es':.25}
SEED = 20261018
# Pilot acquisition is intentionally bounded. Increase only after inspecting audit and rights.
FETCH_WIKIBOOKS = True
WIKIBOOKS_PAGES_PER_LANG = 20
FETCH_WIKIPEDIA = False  # v4 already contains samples; opt-in for additional pages
WIKIPEDIA_PAGES_PER_LANG = 20
REQUEST_DELAY_SECONDS = 0.2
MAX_SOURCE_CHARS = 200_000
MIN_SEGMENT_CHARS = 120
MAX_SEGMENT_CHARS = 4000
MAX_RECORDS_PER_SOURCE = 10000
print('Working directory:', V5)


## 1. Preserve and discover existing files

Existing v4 Wikipedia and Gutenberg downloads are referenced **in place**, not copied or downloaded again. Files under v5 are additional acquisitions. Nothing in this notebook changes the earlier corpora.

In [ ]:
# 3. Existing source inventory

def discover_paths():
    found = {}
    # Priority v5, then v4, then v3. Same logical file is counted once.
    for root in [RAW] + PREVIOUS:
        if not root.exists(): continue
        for p in sorted(root.rglob('*')):
            if not p.is_file() or p.suffix.lower() not in ('.txt', '.jsonl'): continue
            rel = p.relative_to(root)
            if len(rel.parts) < 3 or rel.parts[0] not in LANGS or rel.parts[1] not in DOMAINS: continue
            found.setdefault(rel.as_posix(), p)
    return found

existing = discover_paths()
print('Existing unique source files:', len(existing))
for key, path in sorted(existing.items()):
    print(f'{key:<62} {path.stat().st_size/1024**2:>7.2f} MiB')


## 2. Supplementary educational and language acquisitions

Uses the public MediaWiki API to sample **Wikibooks** (educational) and optionally Wikipedia (language), with page title and revision metadata. Pages are candidates for review, not automatically licensed or approved for training. Network failures are recorded, not hidden.

In [ ]:
# 4. MediaWiki acquisition with metadata and retries
SESSION = requests.Session()
SESSION.headers.update({'User-Agent':'MaiaCorpusResearch/0.3.0 (academic corpus audit; contact via maiaplatform.org)'})
ACQUISITION_ERRORS = []

def api_get(api, params):
    last = None
    for attempt in range(3):
        try:
            response = SESSION.get(api, params=params, timeout=45)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            last = str(exc)
            time.sleep(1.5 * (attempt+1))
    raise RuntimeError(last)

def fetch_wikimedia(lang, project, domain, limit):
    api = f'https://{lang}.{project}.org/w/api.php'
    dest = RAW / lang / domain / project / 'pages.jsonl'
    if dest.exists() and dest.stat().st_size > 0:
        print('Reusing:', dest); return
    rows = []
    try:
        # Page discovery from the main namespace. Wikibooks can include navigation pages:
        # these will be inspected/filtered during curation.
        data = api_get(api, {'action':'query','list':'random','rnnamespace':0,
                             'rnlimit':min(limit,50),'format':'json'})
        titles = [r['title'] for r in data.get('query',{}).get('random',[])]
        for title in titles:
            try:
                obj = api_get(api, {'action':'query','prop':'extracts|revisions|info',
                      'explaintext':1,'exsectionformat':'plain','titles':title,
                      'rvprop':'ids|timestamp','inprop':'url','format':'json','formatversion':2})
                for page in obj.get('query',{}).get('pages',[]):
                    content = page.get('extract','')
                    if len(content.strip()) < MIN_SEGMENT_CHARS: continue
                    rows.append({'text':content[:MAX_SOURCE_CHARS], 'title':page.get('title',title),
                       'url':page.get('fullurl',f'https://{lang}.{project}.org/wiki/{title.replace(" ","_")}'),
                       'revision':(page.get('revisions') or [{}])[0].get('revid'),
                       'retrieved_utc':datetime.now(timezone.utc).isoformat()})
            except Exception as exc:
                ACQUISITION_ERRORS.append({'source':f'{project}-{lang}','title':title,'error':str(exc)})
            time.sleep(REQUEST_DELAY_SECONDS)
        if rows:
            dest.parent.mkdir(parents=True,exist_ok=True)
            with dest.open('w',encoding='utf-8') as f:
                for r in rows: f.write(json.dumps(r,ensure_ascii=False)+'\n')
        print(f'{project}/{lang}: {len(rows)} candidate pages')
    except Exception as exc:
        ACQUISITION_ERRORS.append({'source':f'{project}-{lang}','error':str(exc)})
        print(f'Acquisition failed {project}/{lang}:',exc)

for lang in LANGS:
    if FETCH_WIKIBOOKS: fetch_wikimedia(lang,'wikibooks','educational',WIKIBOOKS_PAGES_PER_LANG)
    if FETCH_WIKIPEDIA: fetch_wikimedia(lang,'wikipedia','language',WIKIPEDIA_PAGES_PER_LANG)
print('Acquisition errors:', len(ACQUISITION_ERRORS))


## 3. Rights ledger — review without automatic approval

`rights_review_v5.json` is editable and persistent. The notebook creates missing records as `pending_review`. **Only a documented manual review** can change a source to `approved` (or `rejected`). Enter reviewer, date, evidence URL, license, and a rationale. Approval is per source file; a blanket approval of all Gutenberg or Wikimedia content is not appropriate.

*No approval is inferred from download success, a dataset-card label, or an empty review field.*

In [ ]:
# 5. Persistent rights ledger and audit trail
LEDGER_PATH = REPORT / 'rights_review_v5.json'

def classify_source(key):
    lang, domain, *rest = key.split('/')
    provider = rest[0] if rest else 'manual'
    if provider == 'wikipedia':
        license_note = 'Wikimedia CC BY-SA/GFDL; page-level exceptions; review required'
        url = f'https://{lang}.wikipedia.org/'
    elif provider == 'wikibooks':
        license_note = 'Wikimedia CC BY-SA/GFDL; page-level exceptions; review required'
        url = f'https://{lang}.wikibooks.org/'
    elif provider == 'gutenberg':
        license_note = 'Public-domain candidate; verify jurisdiction, edition, translation'
        book_id = Path(key).stem
        url = f'https://www.gutenberg.org/ebooks/{book_id}'
    else:
        license_note = 'Unknown; verify source and permission'
        url = ''
    return {'source_id':f'{provider}-{lang}-{Path(key).stem}', 'path':key,
            'lang':lang,'domain':domain,'provider':provider,
            'status':'pending_review','license':license_note,'origin_url':url,
            'review':{'reviewer':'','reviewed_utc':'','evidence_url':'','rationale':''}}

ledger = json.loads(LEDGER_PATH.read_text(encoding='utf-8')) if LEDGER_PATH.exists() else {}
paths = discover_paths()
for key in paths: ledger.setdefault(key,classify_source(key))
LEDGER_PATH.write_text(json.dumps(ledger,indent=2,ensure_ascii=False),encoding='utf-8')
print('Ledger:', LEDGER_PATH)
print('Statuses:',dict(Counter(x.get('status','unknown') for x in ledger.values())))
print('To review: open rights_review_v5.json in Drive and complete entries; then rerun from this cell.')


## 4. Cleaning, segmentation, source quality, duplicates

Compute *potential* data independently of rights status. Remove obvious Gutenberg boilerplate, segment long works into bounded paragraphs, normalize Unicode, and deduplicate by normalized content hash. A Gutenberg dictionary, scripture, and historical prose are flagged for manual genre review instead of being treated as ordinary creative narrative. This is a conservative heuristic, not a substitute for human curation.

In [ ]:
# 6. Parsers, cleaning, segmentation
from transformers import AutoTokenizer
try:
    audit_tokenizer = AutoTokenizer.from_pretrained('gpt2',use_fast=True)
    TOKENIZER_NAME = 'gpt2 (proxy)'
except Exception as exc:
    audit_tokenizer = None
    TOKENIZER_NAME = 'characters-only; tokenizer unavailable'
    print('Tokenizer unavailable:',exc)

GUTENBERG_START = re.compile(r'\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*\*\*\*',re.I)
GUTENBERG_END = re.compile(r'\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*\*\*\*',re.I)

def normalize_text(text):
    text = unicodedata.normalize('NFC',str(text)).replace('\r\n','\n').replace('\r','\n')
    text = re.sub(r'[ \t]+',' ',text)
    return re.sub(r'\n{4,}','\n\n\n',text).strip()

def strip_gutenberg(text):
    a=GUTENBERG_START.search(text)
    b=GUTENBERG_END.search(text)
    if a: text=text[a.end():]
    if b:
        # Re-search end in truncated text, because original offsets may have changed.
        b2=GUTENBERG_END.search(text)
        if b2: text=text[:b2.start()]
    return text

def paragraphs(text):
    # Preserve paragraph boundaries where possible, then split oversized chunks by sentences.
    for block in re.split(r'\n\s*\n',text):
        block=normalize_text(block)
        if not block: continue
        if len(block)<=MAX_SEGMENT_CHARS:
            if len(block)>=MIN_SEGMENT_CHARS: yield block
            continue
        parts=re.split(r'(?<=[.!?])\s+',block)
        current=''
        for part in parts:
            if len(part)>MAX_SEGMENT_CHARS:
                if len(current)>=MIN_SEGMENT_CHARS: yield current
                current=''
                for i in range(0,len(part),MAX_SEGMENT_CHARS):
                    chunk=part[i:i+MAX_SEGMENT_CHARS]
                    if len(chunk)>=MIN_SEGMENT_CHARS: yield chunk
                continue
            if len(current)+len(part)+1>MAX_SEGMENT_CHARS:
                if len(current)>=MIN_SEGMENT_CHARS: yield current
                current=''
            current=(current+' '+part).strip()
        if len(current)>=MIN_SEGMENT_CHARS: yield current

def iter_source_records(path):
    if path.suffix.lower()=='.txt':
        text=path.read_text(encoding='utf-8',errors='replace')
        if '/gutenberg/' in path.as_posix(): text=strip_gutenberg(text)
        yield {'text':text,'title':path.stem,'url':None}
    else:
        with path.open(encoding='utf-8',errors='replace') as f:
            for i,line in enumerate(f):
                if i>=MAX_RECORDS_PER_SOURCE: break
                if not line.strip(): continue
                try:
                    row=json.loads(line)
                    if isinstance(row,dict) and isinstance(row.get('text'),str): yield row
                except json.JSONDecodeError: continue

def sha(text):
    compact=re.sub(r'\s+',' ',unicodedata.normalize('NFKC',text).casefold()).strip()
    return hashlib.sha256(compact.encode('utf-8')).hexdigest()

def token_count(text):
    return len(audit_tokenizer.encode(text,add_special_tokens=False)) if audit_tokenizer else None

GENRE_FLAGS = {'31552':'dictionary','62383':'scripture','26777':'historical/nonfiction'}
print('Tokenizer:',TOKENIZER_NAME)


In [ ]:
# 7. Streamed inventory: source-level statistics and optional reviewed-only export
# Only aggregates and approved data are retained in memory. Avoid storing millions of pending segments.
ledger = json.loads(LEDGER_PATH.read_text(encoding='utf-8'))
paths = discover_paths()
source_stats=[]; reviewed_rows=[]; seen=set(); duplicate_count=0
for key,path in sorted(paths.items()):
    info=ledger.get(key,classify_source(key))
    status=info.get('status','pending_review')
    review=info.get('review') or {}
    complete_review=bool(review.get('reviewer') and review.get('reviewed_utc') and
                         review.get('evidence_url') and review.get('rationale') and
                         info.get('license'))
    # Fail closed if status says approved but review evidence is incomplete.
    eligible = (status=='approved' and complete_review)
    if status=='approved' and not complete_review:
        print('WARNING: incomplete approval ignored:',key)
    st={'source':key,'lang':info['lang'],'domain':info['domain'],'status':status,
        'review_complete':complete_review,'approved_for_training':eligible,
        'bytes':path.stat().st_size,'segments_unique':0,'segments_duplicate':0,
        'characters_unique':0,'tokens_proxy_unique':0,'genre_flag':GENRE_FLAGS.get(path.stem,''),
        'source_path':str(path)}
    for doc in iter_source_records(path):
        for segment in paragraphs(doc['text']):
            digest=sha(segment)
            if digest in seen:
                duplicate_count+=1; st['segments_duplicate']+=1; continue
            seen.add(digest)
            st['segments_unique']+=1
            st['characters_unique']+=len(segment)
            t=token_count(segment)
            if t is not None: st['tokens_proxy_unique']+=t
            if eligible:
                reviewed_rows.append({'text':segment,'lang':info['lang'],'domain':info['domain'],
                    'source_id':info['source_id'],'source_file':key,
                    'title':doc.get('title') or path.stem,
                    'origin_url':doc.get('url') or info.get('origin_url'),
                    'license':info.get('license'),'fingerprint':digest})
    source_stats.append(st)
    print(f"{key[:57]:57} {st['segments_unique']:>6} segments  {st['tokens_proxy_unique']:>9,} proxy tokens  {status}")

stats_df=pd.DataFrame(source_stats)
stats_df.to_csv(REPORT/'source_inventory_v5.csv',index=False)
print('\nPotential proxy tokens:',int(stats_df['tokens_proxy_unique'].sum()))
print('Approved records:',len(reviewed_rows),'Duplicate segments:',duplicate_count)


## 5. Coverage and diversity diagnostics

Report by language/domain and source. Token totals are potential (rights pending) versus reviewed. For scientific interpretation, remember that source diversity matters: five novels do not equal five independent genres; dictionary entries and religious scripture are not representative conversational prose.

In [ ]:
# 8. Coverage tables and plots
from IPython.display import display
if len(stats_df):
    for field in ('tokens_proxy_unique','segments_unique'):
        matrix=stats_df.pivot_table(index='lang',columns='domain',values=field,
             aggfunc='sum',fill_value=0).reindex(index=LANGS,columns=DOMAINS,fill_value=0)
        print('\nPOTENTIAL',field); display(matrix)
    reviewed_stats=stats_df[stats_df['approved_for_training']]
    print('\nAPPROVED proxy tokens:',int(reviewed_stats['tokens_proxy_unique'].sum()))
    print('\nLargest sources by proxy tokens:')
    display(stats_df.sort_values('tokens_proxy_unique',ascending=False)[
      ['source','status','genre_flag','segments_unique','tokens_proxy_unique']].head(25))
    missing_potential=[(l,d) for l in LANGS for d in DOMAINS
      if stats_df[(stats_df.lang==l)&(stats_df.domain==d)]['segments_unique'].sum()==0]
    print('Missing potential language/domain cells:',missing_potential)


## 6. Dialogue and educational source gaps

Do **not** mislabel a narrative book as real dialogue. The notebook can optionally import *licensed dramatic texts* as `dialogue` after manual genre and rights review, but these remain **scripted** and should be reported separately. Contemporary human conversations require a genuinely suitable source with documented permission, privacy review, and provenance.

You can place additional `.txt` or `.jsonl` files in `corpus_v5/inbox/<lang>/<domain>/<provider>/...` and rerun cells 5–8. Each new file automatically receives `pending_review`. JSONL requires a `text` field and may include `title`, `url` and provenance fields.

In [ ]:
# 9. Source diversity & dialogue diagnostics
by_domain=stats_df.groupby('domain',as_index=False).agg(
    sources=('source','count'),segments=('segments_unique','sum'),
    tokens=('tokens_proxy_unique','sum')) if len(stats_df) else pd.DataFrame()
display(by_domain)
print('Dialogue source files:',len(stats_df[stats_df.domain=='dialogue']))
print('Educational source files:',len(stats_df[stats_df.domain=='educational']))
print('WARNING: scripted plays and generated dialogue must never be counted as authentic chat.')


## 7. Source-level splits, export gate and reproducible manifest

A random paragraph split leaks passages from the same work across training and evaluation. This notebook assigns **whole source files** to splits, then checks exact fingerprints. Source-level splits are only a baseline; additional author/work/translation-family leakage controls will be needed for the scientific benchmark.

Approved-only export remains empty until reviews are completed. Synthetic logic from v4 is **not** silently included or relabeled as authentic text.

In [ ]:
# 10. Deterministic source-level split and export
rng=random.Random(SEED)
source_keys=sorted({r['source_file'] for r in reviewed_rows})
rng.shuffle(source_keys)
if len(source_keys)<3:
    # Do not pretend we have an independent evaluation set from one or two works.
    split_map={s:'train' for s in source_keys}
    split_warning='Fewer than three approved sources: independent source-level validation unavailable'
else:
    n=len(source_keys)
    n_val=max(1,round(.10*n)); n_test=max(1,round(.10*n))
    split_map={s:('test' if i<n_test else 'validation' if i<n_test+n_val else 'train')
               for i,s in enumerate(source_keys)}
    split_warning=''

for split in ('train','validation','test'):
    path=OUT/f'authentic_{split}_approved.jsonl'
    with path.open('w',encoding='utf-8') as f:
        for row in reviewed_rows:
            if split_map.get(row['source_file'])==split:
                f.write(json.dumps({k:v for k,v in row.items() if k!='fingerprint'},ensure_ascii=False)+'\n')
    print(split,':',sum(split_map.get(r['source_file'])==split for r in reviewed_rows),'records')
print('Split warning:',split_warning or 'none')


In [ ]:
# 11. Readiness and provenance manifest
source_records=stats_df.to_dict(orient='records') if len(stats_df) else []
potential_tokens=int(stats_df['tokens_proxy_unique'].sum()) if len(stats_df) else 0
approved_tokens=int(stats_df.loc[stats_df.approved_for_training,'tokens_proxy_unique'].sum()) if len(stats_df) else 0
coverage={(l,d):int(stats_df[(stats_df.lang==l)&(stats_df.domain==d)&
                (stats_df.approved_for_training)]['tokens_proxy_unique'].sum())
          for l in LANGS for d in DOMAINS}
missing_approved=[[l,d] for (l,d),count in coverage.items() if count==0]
blockers=[]
if any(v.get('status')=='pending_review' for v in ledger.values()): blockers.append('External rights review incomplete')
if missing_approved: blockers.append('Approved language/domain cells missing')
if sum(v for (l,d),v in coverage.items() if d=='dialogue')<100000:
    blockers.append('Fewer than 100k reviewed authentic dialogue proxy tokens')
if split_warning: blockers.append(split_warning)
if audit_tokenizer is None: blockers.append('Tokenizer proxy unavailable')
else: blockers.append('Final Maia tokenizer not trained; token balance not certified')
blockers.append('Quality, near-duplicate, multilingual and benchmark leakage audits still required')
manifest={'version':'0.3.0','builder':'v5-curation','generated_utc':datetime.now(timezone.utc).isoformat(),
 'seed':SEED,'tokenizer_for_audit':TOKENIZER_NAME,
 'potential_external_tokens_proxy':potential_tokens,'approved_external_tokens_proxy':approved_tokens,
 'approved_documents':len(reviewed_rows),'sources':source_records,
 'rights_statuses':dict(Counter(x.get('status','unknown') for x in ledger.values())),
 'missing_approved_cells':missing_approved,
 'target_domain':TARGET_DOMAIN,'target_language':TARGET_LANG,
 'acquisition_errors':ACQUISITION_ERRORS,'duplicate_segments_rejected':duplicate_count,
 'split_by_source':split_map,'readiness_blockers':blockers,'production_ready':False,
 'files':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('*.jsonl')}}
manifest_path=REPORT/'maia_corpus_v030_manifest_v5.json'
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('Manifest:',manifest_path)
print('PRODUCTION READY:',manifest['production_ready'])
print('Blockers:')
for b in blockers: print(' -',b)


## 8. Next scientific actions

1. Review each source's rights, translations, edition, attribution, and intended model use; document decisions in `reports/rights_review_v5.json`.
2. Add contemporary, properly licensed **dialogue** and **educational** sources in all three languages. Avoid using scraped personal conversations without consent and privacy review.
3. Expand the logical corpus beyond syllogism templates and isolate structural holdouts.
4. Train and freeze a PT/EN/ES tokenizer on a representative, reviewed sample; rerun the token audit.
5. Perform near-duplicate, source-family, language-ID, contamination and quality audits before any full pretraining.

**Do not start the 0.3.0 training while `production_ready` is false.**